# BixBench RL Agent — Google Colab 快速启动

本 notebook 解决「Colab 只支持 .ipynb，不支持整个项目」的问题：
通过在 notebook 内部 `git clone` 整个项目后，再调用项目脚本，  
从而在 **单个 .ipynb** 文件中运行完整的 Python 包项目。

---

## 使用步骤
1. 上传此文件到 [Google Colab](https://colab.research.google.com/)，或直接用 **File → Open notebook → GitHub** 输入本仓库地址打开
2. 依次运行每个 Cell（`Shift+Enter`）
3. 在「Step 2：设置 API Key」处填入你的 API Key

---
**运行模式说明：**

| 模式 | 需要 GPU | 费用 | 说明 |
|------|----------|------|------|
| **API 模式（推荐新手）** | ❌ | 按调用计费 | 用 OpenAI/DeepSeek API 作为 LLM，不加载本地模型 |
| **本地小模型（7B）** | ✅ T4 16GB | 免费 Colab | 加载 Qwen2.5-7B 到 GPU，完全离线推理 |
| **完整训练（30B）** | ✅ A100 80GB | Colab Pro+ | 参照 README 的 GPU 训练步骤 |

## Step 1：克隆项目 & 设置工作目录

> 这一步解决「Colab 只支持 .ipynb」的核心问题：
> 把整个 Python 包克隆到 Colab 的运行环境中，然后把项目根目录加入 `sys.path`。

In [ ]:
import os, sys

REPO_URL = "https://github.com/XUWEI1999-2090/rl-llm-agent.git"
REPO_DIR = "/content/rl-llm-agent"
PROJECT_DIR = f"{REPO_DIR}/bixbench-rl-agent"

# 克隆（如果已存在则跳过）
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print(f"仓库已存在于 {REPO_DIR}，跳过克隆")
    !git -C {REPO_DIR} pull

# 切换到项目子目录
os.chdir(PROJECT_DIR)

# 把 src/ 加入 Python 路径，使 `from src.xxx import yyy` 可用
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f"当前工作目录：{os.getcwd()}")
print(f"项目结构：")
!ls -la

## Step 2：设置 API Key 和 HuggingFace 镜像

> **国内用户必须设置 `HF_ENDPOINT`**，否则访问 HuggingFace 会超时。

In [ ]:
import os

# ── HuggingFace 国内镜像（国内用户必填）────────────────────────────────────
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# ── LLM API Key（二选一）─────────────────────────────────────────────────
# 方案 A：OpenAI（国内可能需要代理）
# os.environ["OPENAI_API_KEY"] = "sk-..."          # 替换成你的 Key

# 方案 B：DeepSeek（国内可直接访问，推荐）
# 注册地址：https://platform.deepseek.com
os.environ["OPENAI_API_KEY"] = "sk-..."             # 替换成你的 DeepSeek Key
os.environ["OPENAI_BASE_URL"] = "https://api.deepseek.com"  # DeepSeek 兼容 OpenAI 接口

# ── 使用的模型名称（与 API 提供商对应）──────────────────────────────────────
# DeepSeek:  "deepseek-chat" 或 "deepseek-coder"
# OpenAI:    "gpt-4o" 或 "gpt-4o-mini"
# 本地 vLLM: 留空，在 Step 5 中配置
API_MODEL_NAME = "deepseek-chat"   # ← 根据你的 API 提供商修改

print("API Key 配置完成")
print(f"HF_ENDPOINT = {os.environ['HF_ENDPOINT']}")
print(f"OPENAI_BASE_URL = {os.environ.get('OPENAI_BASE_URL', '（默认 OpenAI）')}")
print(f"API 模型 = {API_MODEL_NAME}")

## Step 3：安装依赖

Colab 已预装大部分包（torch、transformers 等），这里只安装项目特有依赖。  
**注意：安装 aviary / data-analysis-crow 等需要从源码克隆，会花 3-5 分钟。**

In [ ]:
# 安装基础 PyPI 依赖（快，约 1 分钟）
!pip install -q \
    pydantic>=2.6.0 \
    pyyaml>=6.0.1 \
    httpx>=0.27.0 \
    tqdm>=4.66.0 \
    rich>=13.7.0 \
    datasets>=2.18.0 \
    huggingface-hub>=0.22.0 \
    litellm>=1.30.0 \
    openai>=1.14.0 \
    nbformat>=5.9.0
print("基础依赖安装完成")

In [ ]:
# 安装 FutureHouse Aviary（notebook 环境 + ldp agent 框架）
# 必须从源码安装，PyPI 上的 'aviary' 是另一个项目
import os
AVIARY_DIR = "/content/aviary"
if not os.path.exists(AVIARY_DIR):
    !git clone --depth 1 https://github.com/Future-House/aviary.git {AVIARY_DIR}

!pip install -q -e "{AVIARY_DIR}/packages/aviary[notebook]"
!pip install -q -e "{AVIARY_DIR}/packages/ldp"
print("aviary + ldp 安装完成")

In [ ]:
# 安装 FutureHouse data-analysis-crow（Hypothesis 流水线需要）
import os
CROW_DIR = "/content/data-analysis-crow"
if not os.path.exists(CROW_DIR):
    !git clone --depth 1 https://github.com/Future-House/data-analysis-crow.git {CROW_DIR}

!pip install -q -e "{CROW_DIR}"
print("data-analysis-crow (fhda) 安装完成")

In [ ]:
# 安装 verifiers（GRPO 训练框架）
# 注：Colab Free 没有足够显存跑真正的 GRPO 训练，这里主要用于导入和调试
!pip install -q 'verifiers[all]'
print("verifiers 安装完成")

## Step 4：验证安装 & 加载数据集

用国内镜像加载 `nvidia/Nemotron-RL-bixbench_hypothesis` 数据集。

In [ ]:
# 验证核心模块可以导入
import importlib, sys

checks = [
    ("datasets", "HuggingFace datasets"),
    ("litellm", "LiteLLM"),
    ("openai", "OpenAI SDK"),
]

for module, name in checks:
    try:
        importlib.import_module(module)
        print(f"✓ {name}")
    except ImportError as e:
        print(f"✗ {name}: {e}")

# 验证本项目 src 模块
try:
    from src.dataset.nemotron_dataset import NemotronHypothesisDataset, HypothesisSample
    print("✓ src.dataset.nemotron_dataset")
except ImportError as e:
    print(f"✗ src.dataset.nemotron_dataset: {e}")

try:
    from src.verifiers.protocol import HYPOTHESIS_PROTOCOL
    from src.verifiers.rubric import StepRubric
    print("✓ src.verifiers")
except ImportError as e:
    print(f"✗ src.verifiers: {e}")

In [ ]:
# 加载数据集（使用国内镜像，streaming=True 避免一次性下载全部数据）
# 如果提示需要登录，请先运行：
#   from huggingface_hub import login
#   login(token="hf_...")
from src.dataset.nemotron_dataset import NemotronHypothesisDataset

# streaming=True：按需加载，避免内存不足
# 如果内存充足，可以改为 streaming=False
ds = NemotronHypothesisDataset(split="train", streaming=True)

print(f"数据集加载完成")
if len(ds) > 0:
    sample = ds[0]
    print(f"第 1 条样本：")
    print(f"  capsule_id: {sample.capsule_id}")
    print(f"  hypothesis: {sample.hypothesis[:100]}...")
    print(f"  answer:     {sample.answer}")
else:
    # streaming 模式下 len() 为 0，需要直接迭代
    print("（streaming 模式，直接迭代示例：）")
    for i, sample in enumerate(ds._hf_dataset):
        from src.dataset.nemotron_dataset import NemotronHypothesisDataset
        s = NemotronHypothesisDataset._row_to_sample(sample)
        print(f"  capsule_id: {s.capsule_id}")
        print(f"  hypothesis: {s.hypothesis[:120]}...")
        print(f"  answer:     {s.answer}")
        if i >= 2:
            break

## Step 5：用 API 运行单条样本推理（无 GPU）

用 API 模式（DeepSeek / OpenAI）运行一条完整的 Hypothesis 分析，
不需要本地 GPU，适合**验证环境是否正常**。

> **注意**：完整训练（GRPO）仍需 GPU，见 Step 6。

In [ ]:
import asyncio, os, sys
from src.dataset.nemotron_dataset import NemotronHypothesisDataset
from src.verifiers.protocol import HYPOTHESIS_PROTOCOL
from src.verifiers.rubric import StepRubric

# ── 取一条样本 ──────────────────────────────────────────────────────────────
ds_eager = NemotronHypothesisDataset(split="train", streaming=False)

if len(ds_eager) == 0:
    print("数据集为空，请检查 HF_ENDPOINT 设置和网络连接。")
else:
    sample = ds_eager[0]
    prompt = sample.format_prompt()
    print("=== 生成的 Prompt ===")
    print(prompt[:600], "...")
    print(f"\n正确答案：{sample.answer}")

In [ ]:
# 用 litellm 调用 API（兼容 OpenAI/DeepSeek/各类供应商）
import litellm

if len(ds_eager) > 0:
    sample = ds_eager[0]
    prompt = sample.format_prompt()

    # litellm 自动根据 model 名称选择 provider
    # 如果设置了 OPENAI_BASE_URL，会自动使用 DeepSeek 等兼容接口
    response = litellm.completion(
        model=API_MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a bioinformatics data analysis expert."},
            {"role": "user",   "content": prompt},
        ],
        max_tokens=1024,
        temperature=0.3,
    )

    reply = response.choices[0].message.content
    print("=== LLM 分析结果 ===")
    print(reply[:2000])
    print(f"\n正确答案：{sample.answer}")

## Step 5b：验证 StepRubric 评分

用上面的 LLM 回复跑一次 `StepRubric.score_trajectory()`，检查评分器是否正常工作。

In [ ]:
from src.verifiers.protocol import HYPOTHESIS_PROTOCOL
from src.verifiers.rubric import StepRubric

rubric = StepRubric(protocol=HYPOTHESIS_PROTOCOL)

# 构造一条最小化的轨迹用于测试
mock_trajectory = {
    "capsule_id": sample.capsule_id,
    "steps": [
        {
            "role": "assistant",
            "action": reply,
            "observation": "",
            "log_prob": -1.0,
        }
    ],
    "answer": reply,
    "ground_truth": sample.answer,
}

try:
    scores = rubric.score_trajectory(mock_trajectory)
    print("=== StepRubric 各步评分 ===")
    for step_name, score in scores.items():
        print(f"  {step_name:30s}: {score:.3f}")
    total = sum(scores.values())
    print(f"  {'合计':30s}: {total:.3f}")
except Exception as e:
    print(f"评分时出错（可能需要完整的环境数据）: {e}")

## Step 6：在 Colab GPU 上运行本地小模型（可选）

如果你在 **Colab Free（T4 16GB）** 上运行，可以加载 `Qwen2.5-7B-Instruct`。  
如果没有 GPU 或不需要本地模型，直接跳到 Step 7。

> ⚠️ 此步骤需要 ~15GB 磁盘空间和 ~14GB 显存，Colab Free T4 勉强够用。

In [ ]:
# 检查 GPU
import subprocess
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f"检测到 GPU：{result.stdout.strip()}")
else:
    print("未检测到 GPU，跳过本地模型加载")
    print("如需 GPU：Runtime → Change runtime type → T4 GPU")

In [ ]:
# 加载本地 7B 模型（需要 GPU；如没有 GPU 请跳过此 Cell）
import subprocess, sys

HAS_GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0

if HAS_GPU:
    !pip install -q transformers>=4.39.0 accelerate>=0.27.0

    from transformers import AutoTokenizer, AutoModelForCausalLM
    import torch

    LOCAL_MODEL = "Qwen/Qwen2.5-7B-Instruct"
    print(f"加载模型 {LOCAL_MODEL}（约需 5 分钟）...")

    tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)
    model = AutoModelForCausalLM.from_pretrained(
        LOCAL_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    print(f"模型加载完成，设备：{next(model.parameters()).device}")

    # 用本地模型推理
    if len(ds_eager) > 0:
        sample = ds_eager[0]
        messages = [
            {"role": "system", "content": "You are a bioinformatics data analysis expert."},
            {"role": "user",   "content": sample.format_prompt()},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=512, temperature=0.3, do_sample=True)
        answer = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        print("=== 本地模型输出 ===")
        print(answer)
else:
    print("跳过（无 GPU）")

## Step 7：运行完整训练脚本（调试模式）

在 Colab 中用 `!python` 直接调用项目的训练脚本，
就像在本地终端里运行一样。

这里使用调试配置（n_iterations=2，samples_per_iter=1）快速验证流程正常。

In [ ]:
import yaml, os

# 生成一个轻量调试配置
debug_config = {
    "dataset": {
        "hf_id": "nvidia/Nemotron-RL-bixbench_hypothesis",
        "capsule_data_dir": None,
        "train_split": "train",
        "val_split": "validation",
    },
    "env": {
        "name": "CrowRLEnv",
        "docker_image": "futurehouse/bixbench:aviary-notebook-env",
        "use_docker": False,   # Colab 上禁用 Docker
        "max_steps": 5,
    },
    "model": {
        "name": API_MODEL_NAME,  # 使用 Step 2 中设置的 API 模型
    },
    "grpo": {
        "group_size": 2,          # 调试时减小
        "use_step_level": True,
        "clip_range": 0.2,
        "kl_coef": 0.01,
        "temperature": 0.7,
        "top_p": 0.9,
        "max_new_tokens": 512,
        "max_context_length": 4096,
    },
    "training": {
        "n_iterations": 2,        # 调试时只跑 2 轮
        "samples_per_iter": 1,
        "n_parallel_rollouts": 2,
        "eval_every": 1,
        "learning_rate": 2.0e-5,
        "gradient_accumulation_steps": 1,
        "batch_size": 1,
        "n_epochs": 1,
        "checkpoint_dir": "/content/checkpoints/",
    },
    "protocol": {
        "name": "hypothesis_testing",
        "steps": [
            {"name": "hypothesis_understanding", "weight": 0.10},
            {"name": "data_loading",             "weight": 0.15},
            {"name": "exploratory_analysis",     "weight": 0.20},
            {"name": "statistical_testing",      "weight": 0.25},
            {"name": "interpretation",           "weight": 0.15},
            {"name": "answer_submission",        "weight": 0.15},
        ],
    },
    "logging": {
        "wandb": {"enabled": False},
        "log_dir": "/content/logs/",
    },
}

debug_config_path = "/content/debug_config.yaml"
with open(debug_config_path, "w") as f:
    yaml.dump(debug_config, f, allow_unicode=True)

print(f"调试配置已写入 {debug_config_path}")

In [ ]:
# 运行训练脚本（调试 2 轮）
# 这里通过 !python 直接调用项目里的脚本，
# 与在本地终端运行完全相同
!cd {PROJECT_DIR} && python scripts/train_grpo_hypothesis.py \
    --config /content/debug_config.yaml

## Step 8：保存结果到 Google Drive（可选）

Colab 的 `/content/` 目录在断开连接后会丢失，如需持久化保存：
- checkpoints
- 训练 metrics
- 日志

In [ ]:
# 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

DRIVE_SAVE_DIR = "/content/drive/MyDrive/bixbench-rl-results"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

# 复制 checkpoints 和 logs
for src in ["/content/checkpoints", "/content/logs"]:
    if os.path.exists(src):
        dest = os.path.join(DRIVE_SAVE_DIR, os.path.basename(src))
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print(f"✓ 已保存 {src} → {dest}")
    else:
        print(f"  {src} 不存在，跳过")

print(f"\n结果已保存到 Google Drive: {DRIVE_SAVE_DIR}")

---
## 常见问题 FAQ

**Q: 加载数据集时报 `ConnectionError` 或超时**  
A: 确认 Step 2 中已设置 `os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"`，并在**所有** `import datasets` 之前设置。

**Q: 提示 `401 Unauthorized`**  
A: 数据集需要 HuggingFace 认证。运行：
```python
from huggingface_hub import login
login(token="hf_你的token")  # 在 https://huggingface.co/settings/tokens 获取
```

**Q: `ImportError: No module named 'aviary'`**  
A: 重新运行 Step 3 中「安装 FutureHouse Aviary」的 Cell。

**Q: Colab 因内存不足崩溃（OOM）**  
A: 使用流式加载：`NemotronHypothesisDataset(streaming=True)`，或减小 `max_new_tokens`。

**Q: 想用完整 GPU 训练怎么办？**  
A: 参考 README 的「完整 GPU 训练」章节，或在 AutoDL / 阿里云 PAI 上租用 A100 实例，直接运行 `bash scripts/setup_env.sh`。